# Stochastic Character Mapping

This page demonstrates the method ``simulate_stochastic_map`` for examining transitions in a discrete character across the edges of a phylogeny.

Stochastic character mapping samples complete histories of a discrete trait on a phylogeny. Instead of assigning one state to each node, it samples where state changes may have occurred along every branch, conditional on observed data and a fitted continuous-time Markov chain (CTMC) model.

The method was introduced for mapping substitutions and character changes on trees by Nielsen (2002) and Huelsenbeck, Nielsen, and Bollback (2003), and popularized for discrete-trait analyses by SIMMAP (Bollback 2006). Related CTMC summaries, such as labeled transition counts, can be interpreted as stochastic-map statistics (Minin and Suchard 2008).

Replicate stochastic maps can be used to compute several practical summaries: total branch time spent in each state (dwell time), counts of transitions among states, directional gains and losses, branch-specific probabilities that a transition occurred, and uncertainty in where along the tree transitions are placed.

In [1]:
import numpy as np
import pandas as pd
import toytree

## Setup

A stochastic map requires a tree, observed discrete states, and a fitted Mk model. Here we simulate a small three-state trait only at the tips, then fit an equal-rates model with `fit_discrete_ctmc()`.

The observed tip states are stored to the tree as the node feature `X`. Internal nodes are missing because they will be treated as unknown during fitting and mapping.

In [2]:
# simulate a small tree
tree = toytree.rtree.unittree(ntips=12, treeheight=1.0, seed=123)

# simulate a 3-state trait on the tips
tree.pcm.simulate_discrete_trait(
    nstates=3,
    model="ER",
    name="X",
    state_names="ABC",
    tips_only=True,
    inplace=True,
    seed=2,
)

# fit a CTMC equal-rates model
fit = tree.pcm.fit_discrete_ctmc(data="X", nstates=3, model="ER")

In [3]:
tree.draw(
    width=500,
    height=350,
    node_sizes=10,
    node_mask=(1, 0, 0),
    node_colors=("X", "Set2"),
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="500.0px" height="350.0px" viewBox="0 0 500.0 350.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t081264d0228b4d8ab5dd4de3cec78e01"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

## Sample Maps

`simulate_stochastic_map()` samples one or more full branch histories under the fitted model and returns a `PCMStochasticMapResult` object. 



In [4]:
result = tree.pcm.simulate_stochastic_map(
    data="X",
    model_fit=fit,
    nreplicates=10,
    seed=3,
)

The fundamental result is stored in the `segments` table, where each row in a sampled branch interval that records a state, the edge it belongs to, and its start and end time measured from the child end of that branch. 

Each ``map_id`` is one stochastic replicate, representing one sampled history that can be plotted. The `duration` values for all segments on an edge sum to that branch length. 

In [5]:
columns = ["map_id", "edge_id", "child", "parent", "state", "t_start", "t_end", "duration"]
result.segments[columns].head(12)

,map_id,edge_id,child,parent,state,t_start,t_end,duration
0,0,0,0,14,C,0.000000,0.909091,0.909091
1,0,1,1,13,B,0.000000,0.727273,0.727273
2,0,2,2,12,B,0.000000,0.402457,0.402457
3,0,2,2,12,A,0.402457,0.545455,0.142998
4,0,3,3,12,A,0.000000,0.353753,0.353753
5,0,3,3,12,B,0.353753,0.379754,0.026001
6,0,3,3,12,A,0.379754,0.545455,0.165700
7,0,4,4,15,C,0.000000,0.545455,0.545455
8,0,5,5,15,C,0.000000,0.545455,0.545455
9,0,6,6,16,B,0.000000,0.228353,0.228353


## Visualize a Map

Use `tree.annotate.add_edge_stochastic_map()` to overlay one replicate on an existing tree drawing.

In [7]:
canvas, axes, mark = tree.draw(width=550, height=350)
tree.annotate.add_edge_stochastic_map(
    axes,
    data=result,
    map_id=0,
    color="Set2",
    width=4,
);
tree.annotate.add_tip_markers(axes, color=("X", "Set2"), size=10);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="550.0px" height="350.0px" viewBox="0 0 550.0 350.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t57551f8feb704a27b1263d535fff910a"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

The colored edge segments show the sampled state through time. This is one possible history, not a single best reconstruction. Repeating the mapping samples alternative histories that are also compatible with the model and data.

## Interpret Statistics

In addition to the `segments` table, the `PCMStochasticMapResult` object also stores several summary properties computed across replicates maps. 

### `dwell`
The `dwell` table reports total branch time spent in each state for each map replicate.

In [8]:
result.dwell.head(9)

,map_id,state_idx,state,total_time
0,0,0,A,1.202706
1,0,1,B,2.235467
2,0,2,C,4.743645
3,1,0,A,0.973180
4,1,1,B,2.094468
5,1,2,C,5.114169
6,2,0,A,0.288469
7,2,1,B,2.581000
8,2,2,C,5.312350


### ``dwell_stats``

The `dwell_stats` table summarizes dwell times across replicate simulations.

Dwell time is measured in the same units as branch lengths. Averaging over replicates gives a compact summary of sampled histories, while the variation among replicates shows uncertainty in those histories.

In [9]:
result.dwell_stats.round(3)

,state_idx,state,mean_total_time,sd_total_time,q025_total_time,q50_total_time,q975_total_time,prob_nonzero_total_time
0,0,A,0.711,0.463,0.078,0.850,1.288,1.0
1,1,B,2.354,0.581,1.295,2.408,3.010,1.0
2,2,C,5.117,0.414,4.443,5.130,5.821,1.0


### ``transition_stats``

Transition counts are sampled events, not fitted rate parameters. A large count for one state change means that transition occurred often in the sampled histories. It does not by itself imply a high instantaneous rate unless branch lengths, state frequencies, and model uncertainty are also considered.

In [10]:
result.transition_stats.round(3)

,from_state_idx,to_state_idx,from_state,to_state,mean,sd,q025,q50,q975,prob_nonzero
0,0,1,A,B,1.4,1.075,0.0,1.0,3.000,0.8
1,0,2,A,C,0.6,0.843,0.0,0.0,2.000,0.4
2,1,0,B,A,1.1,0.876,0.0,1.0,2.775,0.8
3,1,2,B,C,1.7,1.252,0.0,2.0,3.775,0.8
4,2,0,C,A,1.3,0.949,0.0,2.0,2.000,0.7
5,2,1,C,B,3.3,0.483,3.0,3.0,4.000,1.0


### ``edge_transition_stats``

This table summarizes all possible transition events on each edge of the tree across all simulations. It provides stats to investigate questions like "how frequently did state A change to B on branch 10?". 

In [11]:
result.edge_transition_stats.head(10)

,edge_id,child,parent,from_state_idx,to_state_idx,from_state,to_state,mean,sd,q025,q50,q975,prob_nonzero
0,0,0,14,0,1,A,B,0.1,0.316228,0.0,0.0,0.775,0.1
1,0,0,14,0,2,A,C,0.1,0.316228,0.0,0.0,0.775,0.1
2,0,0,14,1,0,B,A,0.0,0.000000,0.0,0.0,0.000,0.0
3,0,0,14,1,2,B,C,0.3,0.483046,0.0,0.0,1.000,0.3
4,0,0,14,2,0,C,A,0.0,0.000000,0.0,0.0,0.000,0.0
5,0,0,14,2,1,C,B,0.0,0.000000,0.0,0.0,0.000,0.0
6,1,1,13,0,1,A,B,0.3,0.483046,0.0,0.0,1.000,0.3
7,1,1,13,0,2,A,C,0.1,0.316228,0.0,0.0,0.775,0.1
8,1,1,13,1,0,B,A,0.0,0.000000,0.0,0.0,0.000,0.0
9,1,1,13,1,2,B,C,0.0,0.000000,0.0,0.0,0.000,0.0


## Replicates and `map_id`

Increasing `nreplicates` gives more sampled histories. The `map_id` key is also useful for further examining statistics across replicates.

In [12]:
replicate_dwell = (
    result.dwell.pivot(index="map_id", columns="state", values="total_time")
    .round(3)
)
replicate_dwell

state,A,B,C
map_id,,,
0,1.203,2.235,4.744
1,0.973,2.094,5.114
2,0.288,2.581,5.312
3,1.313,1.908,4.961
4,0.893,2.144,5.145
5,0.226,2.663,5.293
6,0.271,2.802,5.108
7,1.097,1.117,5.968
8,0.807,3.019,4.356


Also use `map_id` to select which replicate to draw. Compared to the drawing above, this uses the same tree, observations, and fitted model, but a different `map_id`. Differences among replicates represent uncertainty in where transitions occurred and which states occupied unsampled parts of the tree.

In [13]:
canvas, axes, mark = tree.draw(width=550, height=350)
tree.annotate.add_edge_stochastic_map(
    axes,
    data=result,
    map_id=1,
    color="Set2",
    width=4,
);
tree.annotate.add_tip_markers(axes, color=("X", "Set2"), size=10);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="550.0px" height="350.0px" viewBox="0 0 550.0 350.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t56a381ab4d5b44f58e271b794d60e49e"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

## Branch-Specific Gains

A common use of stochastic maps is to estimate how often a particular transition occurred on a particular branch. In this example a rare derived trait appears in two separated clades, suggesting that it evolved twice. We will ask how often stochastic maps place a transition to the derived state on the branch subtending one of those clades.

We compare an `ER` model to an `ARD` model. The `ARD` model allows asymmetric transition rates. For binary data this is a gain-loss model with different forward and reverse rates.

In [14]:
# set derived state in two clades
derived_tips = {"r2", "r3", "r10", "r11"}
btree = tree.set_node_data(
    feature="rare_trait",
    data={i: "derived" if i in derived_tips else "ancestral" for i in tree.get_tip_labels()},
    default=np.nan,
)

# select a focal edge
target_node = btree.get_mrca_node("r2", "r3")

In [15]:
# set vectors of edge colors and widths in node idx order
edge_colors = ["darkorange" if node.idx == target_node.idx else "black" for node in btree]
edge_widths = [5 if node.idx == target_node.idx else 2 for node in btree]

# draw the tree with tip states and focal edge highlighted
canvas, axes, mark = btree.draw(
    width=550,
    height=350,
    node_sizes=10,
    node_mask=(1, 0, 0),
    node_colors=("rare_trait", "Set2"),
    edge_colors=edge_colors,
    edge_widths=edge_widths,
)

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="550.0px" height="350.0px" viewBox="0 0 550.0 350.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t5aebbba8ed254fce81821c9a207022d4"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11

The highlighted edge is the branch subtending tips `r2` and `r3`. We will estimate how many sampled maps include a gain from `ancestral` to `derived` on this edge.

In [16]:
# fit a model and simulate maps
fit = btree.pcm.fit_discrete_ctmc("rare_trait", nstates=2, model="ER")
smap = btree.pcm.simulate_stochastic_map("rare_trait", model_fit=fit, nreplicates=200, seed=12)

# extract stats for the specified edge and transition of interest
row = smap.edge_transition_stats.query(
    "edge_id==@target_node.idx "
    "and from_state=='ancestral' "
    "and to_state=='derived'"
).iloc[0]

# store result
er_result = {
    "model": fit.model, 
    "log_likelihood": fit.log_likelihood,
    "mean_gains_on_branch": row['mean'], 
    "prob_any_gain_on_branch": row['prob_nonzero'],
}

Next we do the same thing but fitting an "ARD" model:

In [17]:
# fit a model and simulate maps
fit = btree.pcm.fit_discrete_ctmc("rare_trait", nstates=2, model="ARD")
smap = btree.pcm.simulate_stochastic_map("rare_trait", model_fit=fit, nreplicates=200, seed=12)

# extract stats for the specified edge and transition of interest
row = smap.edge_transition_stats.query(
    "edge_id==@target_node.idx "
    "and from_state=='ancestral' "
    "and to_state=='derived'"
).iloc[0]

# store result
ard_result = {
    "model": fit.model, 
    "log_likelihood": fit.log_likelihood,
    "mean_gains_on_branch": row['mean'], 
    "prob_any_gain_on_branch": row['prob_nonzero'],
}

### Result

The final column estimates the conditional frequency, under each fitted model, that at least one gain to the derived state occurred on the target branch. This quantity is computed from the replicate stochastic maps through `edge_transition_stats`. Changing the model can change this branch-specific conclusion because the fitted model changes the probabilities of gains, losses, and node states that condition the sampled histories.

In [18]:
pd.DataFrame([er_result, ard_result]).round(3)

,model,log_likelihood,mean_gains_on_branch,prob_any_gain_on_branch
0,ER,-7.881,0.515,0.515
1,ARD,-7.190,0.445,0.440


Here we can see the probability of a transition to the derived state on branch ancestral to tips "r2" and "r3" is higher under the "ER" model than the "ARD" model. But in both it is only around 50% probability. This suggests that we can not state with high confidence that a single transition occurred in their ancestor. Instead, transitions could have occurred independently on the long branches leading to each tip, or in a different ancestor. We could compute and plot the probabilities of this transition on each of those branches if we cared to examine it more explicitly.

## Posterior Constraints

In addition to providing a fixed state for each tip in the tree, you can also fix the states of one or more internal nodes of the tree, or set them to non-fixed states, as posterior probability vectors, such as those returned by ancestral-state reconstruction. 

In this mode, non-missing entries must all be vectors of length `nstates` that sum to 1. One-hot vectors behave as fixed state constraints, while non-one-hot vectors are sampled from their probabilities.

Posterior vectors are ordered by the fitted model states. When maps are sampled from vectors only, the `state` column reports state indices rather than the original tip-state labels.

In [19]:
# fit a CTMC model and infer ancestral state posterior probabilities
asr = tree.pcm.infer_ancestral_states_discrete_ctmc(
    data="X",
    nstates=3,
    model="ER",
)
posteriors = asr["data"]["X_anc_posterior"]
posteriors

0                                       (0.0, 0.0, 1.0)
1                                       (0.0, 1.0, 0.0)
2                                       (0.0, 1.0, 0.0)
3                                       (1.0, 0.0, 0.0)
4                                       (0.0, 0.0, 1.0)
5                                       (0.0, 0.0, 1.0)
6                                       (0.0, 1.0, 0.0)
7                                       (0.0, 1.0, 0.0)
8                                       (0.0, 0.0, 1.0)
9                                       (0.0, 0.0, 1.0)
10                                      (0.0, 0.0, 1.0)
11                                      (0.0, 0.0, 1.0)
12    (0.22975007401102754, 0.5507508602960469, 0.21...
13    (0.1375626617872403, 0.5240922810395716, 0.338...
14    (0.07247423302851404, 0.3176720647501351, 0.60...
15    (0.026724406205176465, 0.10342172261557021, 0....
16    (0.033496920790761914, 0.22666041288207353, 0....
17    (0.000902272121284814, 0.00127964791859120

In [20]:
# draw the tree node state posteriors
canvas, axes, mark = btree.draw(width=550, height=350)
btree.annotate.add_node_pie_markers(
    axes, data=posteriors, 
    colors="Set2", size=15, 
    istroke="black", istroke_width=1,
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="550.0px" height="350.0px" viewBox="0 0 550.0 350.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t0191f524443a4c2ea0808d64957750f6"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 0, 0, 1 0, 1, 0 0, 1, 0 1, 0, 0 0, 0, 1 0, 0, 1 0, 1, 0 0, 1, 0 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0.22975, 0.550751, 0.219499 0.137563, 0.524092, 0.338345 0.0724742, 0.317672, 0.609854 0.0267244, 0.103422, 0.869854 0.0334969, 0.22666, 0.739843 0.000902272, 0.00127965, 0.997818 0.00373475, 0.00914674, 0.987119 0.0123095, 0.0505467, 0.937144 0.0286918, 0.20781, 0.763498 0.0336265, 0.239423, 0.726951 0.0557921, 0.278537, 0.665671

In [21]:
posterior_result = tree.pcm.simulate_stochastic_map(
    data=posteriors,
    model_fit=asr["model_fit"],
    nreplicates=2,
    seed=4,
)

posterior_result.segments[columns].head(10)

,map_id,edge_id,child,parent,state,t_start,t_end,duration
0,0,0,0,14,2,0.000000,0.909091,0.909091
1,0,1,1,13,1,0.000000,0.727273,0.727273
2,0,2,2,12,1,0.000000,0.536811,0.536811
3,0,2,2,12,2,0.536811,0.545455,0.008644
4,0,3,3,12,0,0.000000,0.528509,0.528509
5,0,3,3,12,2,0.528509,0.545455,0.016946
6,0,4,4,15,2,0.000000,0.332101,0.332101
7,0,4,4,15,0,0.332101,0.384472,0.052371
8,0,4,4,15,1,0.384472,0.545455,0.160983
9,0,5,5,15,2,0.000000,0.072761,0.072761


Posterior-vector input is useful when you want stochastic maps conditioned on inferred node uncertainty, or a fixed internal node state, rather than only on scalar observed states at the tips. Do not mix scalar states and posterior vectors in one call; use one input mode for all non-missing entries.

## Related APIs

- `tree.pcm.simulate_stochastic_map()` samples stochastic-map results with segment, event, dwell-time, transition-count, node-state, and branch-specific summary tables.
- `tree.annotate.add_edge_stochastic_map()` draws one replicate on an existing tree plot.
- `tree.pcm.fit_discrete_ctmc()` fits the Mk model used for mapping.
- `tree.pcm.infer_ancestral_states_discrete_ctmc()` estimates node-state posteriors that can be used as mapping constraints.
- `tree.pcm.simulate_discrete_trait()` creates example discrete traits for testing workflows.

## References

- Nielsen, R. 2002. Mapping mutations on phylogenies. *Systematic Biology* 51:729-739. https://doi.org/10.1080/10635150290102393
- Huelsenbeck, J. P., Nielsen, R., and Bollback, J. P. 2003. Stochastic mapping of morphological characters. *Systematic Biology* 52:131-158. https://doi.org/10.1080/10635150390192780
- Bollback, J. P. 2006. SIMMAP: stochastic character mapping of discrete traits on phylogenies. *BMC Bioinformatics* 7:88. https://doi.org/10.1186/1471-2105-7-88
- Minin, V. N., and Suchard, M. A. 2008. Counting labeled transitions in continuous-time Markov models of evolution. *Journal of Mathematical Biology* 56:391-412. https://doi.org/10.1007/s00285-007-0120-8
